<a href="https://colab.research.google.com/github/nivethithanm/torchcode-solutions/blob/main/P0_07_nn_module.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# P0-07: Custom nn.Module

**Difficulty**: 🟢 Easy

**Objective**: Build layers the way PyTorch expects — this is the pattern for EVERY TorchCode problem.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## Task 1: Implement a Linear Layer from Scratch

In [7]:
class MyLinear(nn.Module):
    """Linear layer: y = x @ W.T + b

    Args:
        in_features: input dimension
        out_features: output dimension
        bias: whether to include a bias term
    """
    def __init__(self, in_features: int, out_features: int, bias: bool = True):
        super().__init__()
        # TODO: Create weight as nn.Parameter — shape (out_features, in_features)
        # TODO: Create bias as nn.Parameter — shape (out_features,) or None
        self.weight = nn.Parameter(torch.rand(out_features, in_features), requires_grad=True)
        self.bias = nn.Parameter(torch.zeros(out_features), requires_grad=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: Compute x @ W.T + b
        y = x @ self.weight.transpose(1, 0) + self.bias
        return y

# Test
layer = MyLinear(64, 128)
x = torch.randn(2, 10, 64)
y = layer(x)
assert y.shape == (2, 10, 128)

# Verify it has the right parameters
params = dict(layer.named_parameters())
assert 'weight' in params and params['weight'].shape == (128, 64)
assert 'bias' in params and params['bias'].shape == (128,)
print('✅ Task 1 passed!')

✅ Task 1 passed!


## Task 2: Two-Layer MLP with ReLU

In [8]:
class MyMLP(nn.Module):
    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        # TODO: Two linear layers + store them so parameters are tracked
        self.linear1 = MyLinear(d_model, d_ff)
        self.linear2 = MyLinear(d_ff, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: linear1 → relu → linear2
        y = self.linear1(x)
        y = F.relu(y)
        y = self.linear2(y)
        return y

mlp = MyMLP(64, 256)
x = torch.randn(2, 10, 64)
y = mlp(x)
assert y.shape == (2, 10, 64)

# Count parameters
num_params = sum(p.numel() for p in mlp.parameters())
expected = 64*256 + 256 + 256*64 + 64  # two linear layers with bias
assert num_params == expected, f'Expected {expected} params, got {num_params}'
print(f'Total parameters: {num_params:,}')
print('✅ Task 2 passed!')

Total parameters: 33,088
✅ Task 2 passed!
